In [ ]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


# 02 · RFM 与会员画像
> 目标：把会员切成 8 个可运营的客群，并给出每个客群的画像特征。

In [ ]:
from retailinsight.io.loader import read_parquet
from retailinsight.settings import MART_PROFILE, MART_RFM

rfm = read_parquet(MART_RFM)
profile = read_parquet(MART_PROFILE)
print(f"mart_rfm: {rfm.shape}  mart_profile: {profile.shape}")


## 2.1 RFM 打分口径
> 五分位打分 1~5，≥4 记「高」，再按 2×2×2 映射成 8 类客群。

In [ ]:

print(rfm[["recency_days", "frequency", "monetary", "r_score", "f_score", "m_score",
           "rfm_segment"]].describe().round(2))


## 2.2 八类客群分布与 GMV 贡献

In [ ]:
summary = (rfm.groupby("rfm_segment")
             .agg(members=("member_id", "count"), gmv=("monetary", "sum"),
                  avg_recency=("recency_days", "mean"), avg_frequency=("frequency", "mean"),
                  avg_monetary=("monetary", "mean"))
             .round(2))
summary["member_share"] = (summary["members"] / len(rfm)).round(4)
summary["gmv_share"] = (summary["gmv"] / summary["gmv"].sum()).round(4)
summary.sort_values("gmv", ascending=False)


## 2.3 客群特征对比（画像宽表）
> 画像宽表已 join 流失概率（第 20 个 P0 标签），`churn_covered` 标注是否被覆盖。

In [ ]:
cols = ["avg_basket", "monthly_amount", "yearly_freq", "items_per_txn",
        "category_entropy", "discount_sensitivity", "weekend_ratio"]
cmp = profile.groupby("rfm_segment")[cols].mean().round(3)
cmp.join(summary[["members", "gmv_share"]])


## 2.4 流失概率覆盖率说明
> 流失得分只覆盖末期 cohort 观察窗内的活跃会员，其余为 NaN（保留而非填均值）。

In [ ]:
if "churn_prob" in profile.columns:
    cov = profile["churn_covered"].mean()
    print(f"churn_prob 覆盖率: {cov:.1%}（未覆盖 {int((~profile['churn_covered']).sum()):,} 人保留 NaN）")
    print(profile.groupby("rfm_segment")["churn_prob"].mean().round(4).sort_values(ascending=False))
else:
    print("未生成 churn_prob，请先运行 python -m retailinsight.cli run --stage profile")


## 2.5 分群雷达图

In [ ]:
from retailinsight.viz import charts

print(charts.segment_radar(profile, by="rfm_segment"))
